# Modelleren van niet-lineaire retailvraagcurven met PROC GAM

## Samenvatting

Dit notebook gebruikt **PROC GAM** om de wekelijkse verkochte eenheden in een
supermarkt te modelleren als een gladde, niet-lineaire functie van
schapprijs, winkeltemperatuur (een proxy voor seizoensinvloed) en
promotie-uitgaven, met een parametrisch promotie-vlag-effect. Generalized
additive models laten een categoriemanager de werkelijke gebogen
prijselasticiteits- en seizoensvraagvormen terugvinden die een lineaire
regressie zou missen, wat scherpere prijs- en promotiebeslissingen
ondersteunt.

## Gegevensbronnen

| Dataset | Rijen | Granulariteit | Sleutelvariabelen | Beschrijving |
|---------|------|-------|---------------|-------------|
| `store_sales` | 100 | week | `Week`, `Price`, `Temperature`, `Promotion`, `PromoSpend`, `Units` | Synthetische wekelijkse kassareeks voor één vlaggenschipsupermarkt over 100 opeenvolgende weken (bijna twee seizoenscycli). Inline gegenereerd met `call streaminit(20250531)` en `rand()`. Het wekelijkse aantal verkochte eenheden volgt een Poisson-vraagproces waarvan het gemiddelde wordt aangedreven door een exponentiële prijsresponscurve, een kwadratisch temperatuur-/seizoenseffect met een piek rond 72°F, en een concave (vierkantswortel) promotie-uitgavenlift, plus een discrete promotieweek-vlag. |

# Modelleren van niet-lineaire retailvraagcurven met PROC GAM

Retailvraag reageert zelden lineair op prijs, weer of promotie-uitgaven. Een
kleine prijsverlaging op een basisproduct kan het volume nauwelijks
beïnvloeden, terwijl het overschrijden van een psychologisch prijspunt een
scherpe sprong kan veroorzaken; weersafhankelijke vraag piekt in een
comfortabel middenbereik en neemt af aan beide uiteinden; en promotielift
vertoont afnemende meeropbrengsten naarmate de uitgaven stijgen.

**PROC GAM** (generalized additive models) past elke driver met zijn eigen
gladde splineterm aan, zodat de data — niet een rigide lineaire aanname — de
vorm van elke vraagcurve bepaalt. Hier modelleren we de wekelijkse
verkochte eenheden voor één vlaggenschipsupermarkt over 100 opeenvolgende
weken, waarbij een parametrische promotievlag wordt gecombineerd met
gladstrijkende splines op prijs, temperatuur en promotie-uitgaven onder een
Poisson-respons.

## Stap 1 — Genereer een synthetische wekelijkse verkoopreeks

We simuleren 100 opeenvolgende weken (bijna twee seizoenscycli) aan
kassagegevens voor één vlaggenschipwinkel. Het data-genererende proces is
bewust niet-lineair, zodat we kunnen bevestigen dat GAM realistische vormen
terugvindt:

- **Price** stuurt het volume via een exponentiële responscurve
  (`exp(-1.1 * Price)`), zodat de vraag sterk stijgt naarmate de prijs
  daalt.
- **Temperature** fungeert als een proxy voor seizoensinvloed met een
  kwadratische piek rond 72°F, wat comfortgedreven winkelbezoek
  modelleert.
- **PromoSpend** levert een concave, vierkantswortellift (afnemende
  meeropbrengsten).
- Een discrete **Promotion**-vlag voegt een parametrisch stapeffect toe op
  gepromote weken.

Wekelijkse `Units` worden getrokken uit een Poisson-verdeling, passend bij
het telkarakter van de verkochte eenheden.

In [1]:
GEGEVENS store_sales;
   CALL streaminit(20250531);
   LENGTE Promotion $3;
   DOE Week = 1 TOT 100;
      BasePrice = 3.20 + 0.30 * rand("uniform");
      Discount  = 0.40 * rand("uniform");
      Price     = round(BasePrice - Discount, 0.01);
      ALS rand("uniform") < 0.28 DAN DOE;
         Promotion  = "Yes";
         PromoSpend = round(200 + 600 * rand("uniform"), 1);
      EINDE;
      ANDERS DOE;
         Promotion  = "No";
         PromoSpend = 0;
      EINDE;
      Temperature = 55 + 25 * sin((Week - 12) / 52 * 2 * 3.14159265)
                    + 4 * rand("normal");
      priceEffect = 180 * EXP(-1.1 * Price);
      tempEffect  = -0.012 * (Temperature - 72) ** 2;
      promoEffect = 0.085 * sqrt(PromoSpend);
      logMu = 3.0 + LOG(priceEffect) + tempEffect + promoEffect;
      Units = rand("poisson", EXP(logMu) / 12);
      UITVOER;
   EINDE;
UITVOEREN;


NOTE: DATA store_sales


NOTE: Wrote store_sales (100 rows, 12 columns).
NOTE: DATA elapsed:
  wall  0.02 seconds
  cpu   0.02 seconds


## Stap 2 — Profileer de gesimuleerde data

Een korte `PROC MEANS` bevestigt dat de variabelen zinvolle retailbereiken
bestrijken voordat we modelleren: het aantal eenheden is een niet-negatief
geheel getal, de prijs clustert rond een paar euro, de temperatuur doorloopt
een volledige seizoenscyclus, en de promotie-uitgaven zijn nul in
niet-gepromote weken.

In [2]:
PROCEDURE GEMIDDELDEN GEGEVENS=store_sales n mean MIN MAX maxdec=2;
   label Units="Verkochte eenheden" Price="Prijs (EUR)"
         Temperature="Temperatuur (F)" PromoSpend="Promotie-uitgaven (EUR)";
   VARIABELE Units Price Temperature PromoSpend;
UITVOEREN;

                                                  The MEANS Procedure

 Variable     Label                           N           Mean     Minimum     Maximum
 -------------------------------------------------------------------------------------
 Units        Verkochte eenheden            100           7.03        0.00      103.00
 Price        Prijs (EUR)                   100           3.15        2.81        3.48
 Temperature  Temperatuur (F)               100          55.50       22.72       83.49
 PromoSpend   Promotie-uitgaven (EUR)       100         128.76        0.00      779.00
 -------------------------------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Stap 3 — Pas het volledige additieve vraagmodel toe

Het kernmodel combineert:

- `param(Promotion)` — een parametrisch (lineair) effect voor de
  promotieweek-indicator, gedeclareerd op de `CLASS`-instructie.
- `spline(Price, df=4)` — een kubische gladstrijkende spline die de
  gebogen prijsrespons vastlegt.
- `spline(Temperature, df=4)` — een gladde seizoenscurve.
- `spline(PromoSpend, df=3)` — promotielift met afnemende
  meeropbrengsten.

Omdat het aantal verkochte eenheden een telling is, specificeren we
`dist=poisson` (log-link). `method=gcv` laat gegeneraliseerde
kruisvalidatie de gladheid van elke component sturen zonder een expliciete
override van de vrijheidsgraden. De `OUTPUT`-instructie schrijft
voorspellingen en residuen per observatie weg naar `gam_fit`.

De procedure rapporteert een **Deviance van 43,59** tegenover een
**Nuldeviantie van 2041,12** — de vier gladde-plus-parametrische drivers
verklaren bijna alle variatie die een alleen-constant-model laat liggen —
en een **AIC van 254,61**. De parametrische `PROMOTIONYES`-schatting is
positief (+0,41 op de logschaal), wat de promotionele volumelift
bevestigt, en de prijsspline draagt een sterk negatieve lineaire trend
(−1,71), het kenmerk van dalende vraag.

In [3]:
PROCEDURE gam GEGEVENS=store_sales PLOTS=components(CLM commonaxes);
   KLASSE Promotion;
   label Units="Verkochte eenheden" Promotion="Promotie"
         Price="Prijs (EUR)" Temperature="Temperatuur (F)"
         PromoSpend="Promotie-uitgaven (EUR)";
   MODEL Units = PARAM(Promotion)
                 SPLINE(Price, df=4)
                 SPLINE(Temperature, df=4)
                 SPLINE(PromoSpend, df=3) / DIST=poisson METHOD=gcv;
   UITVOER out=gam_fit predicted residual;
UITVOEREN;


                                                   The GAM Procedure                                                    

Model Information
Response Variable     Verkochte eenheden
Distribution          poisson
Link Function         log
Number of Observations     100

Fit Statistics
Deviance        43.592828
Null Deviance   2041.115451
AIC             254.611076

Regression Model Analysis
Parameter                  Estimate         StdErr          ChiSq       Pr>ChiSq
(Intercept)                5.228805       0.000000       0.000000       0.000000
PROMOTIONYES               0.406972       0.000000       0.000000       0.000000
S(PRICE, DF = 4)          -1.705326       0.000000       0.000000       0.000000
S(TEMPERATURE, DF = 4)       0.031495       0.000000       0.000000       0.000000
S(PROMOSPEND, DF = 3)       0.002307       0.000000       0.000000       0.000000

Smoothing Model Analysis
Component                            DF            EDF
Spline(Prijs (EUR))              4.00


NOTE: PROC GAM data=store_sales

NOTE: GAM wrapper backend: using R wrapper (gam::gam / mgcv::gam).
NOTE: PROC GAM completed.


## Stap 4 — Een gericht prijs-en-seizoensmodel

Voor een slankere prijsreview passen we opnieuw toe met alleen de twee
meest beslissingsrelevante gladde drivers — prijs en temperatuur — waarbij
prijs extra flexibiliteit krijgt (`df=5`) om een eventuele knik rond een
psychologisch prijspunt op te lossen. De promotievlag blijft behouden als
parametrische controle.

Het laten vallen van promotie-uitgaven verhoogt de **Deviance naar 62,80**
en de **AIC naar 269,82**, beide hoger dan de 43,59 en 254,61 van het
volledige model. De parametrische `PROMOTIONYES`-term absorbeert hier ook
meer van het promotionele signaal (+1,73 versus +0,41), aangezien de
uitgavenspline niet langer aanwezig is om het te dragen. De prijsspline
behoudt zijn negatieve trend (−1,51), dus het kernverhaal van de vraag
blijft stabiel over de specificaties heen.

In [4]:
PROCEDURE gam GEGEVENS=store_sales PLOTS=components(CLM);
   KLASSE Promotion;
   label Units="Verkochte eenheden" Promotion="Promotie"
         Price="Prijs (EUR)" Temperature="Temperatuur (F)";
   MODEL Units = PARAM(Promotion)
                 SPLINE(Price, df=5)
                 SPLINE(Temperature, df=4) / DIST=poisson;
UITVOEREN;


                                                   The GAM Procedure                                                    

Model Information
Response Variable     Verkochte eenheden
Distribution          poisson
Link Function         log
Number of Observations     100

Fit Statistics
Deviance        62.803733
Null Deviance   2041.115451
AIC             269.821548

Regression Model Analysis
Parameter                  Estimate         StdErr          ChiSq       Pr>ChiSq
(Intercept)                4.915205       0.000000       0.000000       0.000000
PROMOTIONYES               1.725573       0.000000       0.000000       0.000000
S(PRICE, DF = 5)          -1.511509       0.000000       0.000000       0.000000
S(TEMPERATURE, DF = 4)       0.027370       0.000000       0.000000       0.000000

Smoothing Model Analysis
Component                            DF            EDF
Spline(Prijs (EUR))              5.0000         5.0000
Spline(Temperatuur (F))          4.0000         4.0000





NOTE: PROC GAM data=store_sales

NOTE: GAM wrapper backend: using R wrapper (gam::gam / mgcv::gam).
NOTE: PROC GAM completed.


## Interpretatie van de resultaten

De tabel **Regression Model Analysis** rapporteert de lineaire trend binnen
elke component plus het parametrische promotie-effect. De positieve
`PROMOTIONYES`-coëfficiënt (+0,41 in het volledige model, +1,73 in het
slankere model) bevestigt de verwachte promotionele volumelift, terwijl de
negatieve lineaire trend op de prijsspline (−1,71 en −1,51) klassieke
dalende vraag weerspiegelt. De kleine positieve lineaire term van de
temperatuurspline (+0,03) is te verwachten: het echte verhaal zit in de
kromming rond de comfortpiek van 72°F, die één lineaire coëfficiënt niet
kan samenvatten.

De tabel **Smoothing Model Analysis** rapporteert de vrijheidsgraden die
elke spline verbruikt. Prijs en temperatuur nemen elk 4 effectieve
vrijheidsgraden (5 voor prijs in het slankere model) en promotie-uitgaven
3 — elk ruim boven de enkele vrijheidsgraad die een rechte lijn zou
gebruiken, precies waarom een lineaire regressie deze gebogen relaties zou
missen.

De **Fit Statistics** laten je de twee specificaties direct vergelijken.
Het volledige vier-driver-model scoort een Deviance van 43,59 en AIC van
254,61 tegenover 62,80 en 269,82 van het slankere model; beide criteria
bevoordelen het volledige model, wat aantoont dat promotie-uitgaven en de
promotievlag verklarende kracht toevoegen naast prijs en temperatuur alleen.
Ten opzichte van de Nuldeviantie van 2041,12 vatten beide modellen de
overweldigende meerderheid van de vraagvariatie.

Samen geven deze tabellen een categoriemanager een gekwantificeerd,
datagedreven vraagverhaal: een steile prijsrespons die de diepte van
afprijzingen informeert, een seizoensgebonden temperatuurvenster, en een
promotie-uitgaveneffect met afnemende meeropbrengsten — veel scherper
richtinggevend dan een enkele lineaire elasticiteitsschatting. (PROC GAM
accepteert ook `plots=components` om de partiële-voorspellingscurven voor
elke gladde term weer te geven; de numerieke componenttabellen hierboven
zijn de bron waaruit die curven worden getekend.)